código feito pelo apresentador da aula.

inicialmente são feitas as importações das bibliotecas que serão usadas, da mesma forma que no exemplo 1, mas com algumas bibliotecas adicionais.

In [1]:
import enum
import hashlib
import re
from typing import Any

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_validator,
    model_validator,
    SecretStr,
    ValidationError,
)

Usa-se validações regex (regular expressions) para validar a senha e o nome inserido. O re.compile serve para definir modelos para os textos. Para o VALID_PASSWORD_REGEX, é definido que precisa ter pelo menos uma letra minuscula, maiúscula e 8 caracteres. Para o VALID_NAME_REGEX, define que precisa ter apenas letras e pelo menos 2.

In [2]:
VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

Na classe Role, agora usa potências de 2 para numerar os papéis.

In [3]:
class Role(enum.IntFlag):
    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8

Agora vem a grande diferença entre este exemplo e o anterior. No anterior, são usados somente as validações já existentes no BaseModel, mas agora iremos adicionair mais validações. As primeiras linhas são idênticas ao do exemplo anterior, a diferença se dá a partir do "@field_validator("name)". Essa expressão serve para indicar que a validação a seguir (validate_name) deve ser aplicada ao campo "name". Já existia uma verificação nativa do BaseModel, mas existe a opção de adicionar novas verificações. Neste caso, adicionou a verificação existente no "VALID_NAME_REGEX" que foi definido anteriormente neste código, então ao executar os testes, essa validação também será incluida no campo "name". Em seguida, usa também o fiel_validator para adicionar uma verificação ao campo "role", mas dessa vez, usando o mode = "before". Com o mode before, é possível fazer manipulações no valor de role antes de verificar se o dado é de fato válido ou não, pois é possível adaptar a informação, exemplo: o role aceita o número 4, mas o valor informado é "4". Seria considerado um valor incorreto, mas com o tratamento de informações do before é possível adaptar para ficar da forma correta. O model_validator segue a mesma lógica, a diferença é que ele pode adicionar verificações a todos os campos de uma vez só, enquanto o field_validor verifica apenas um.

In [4]:
class User(BaseModel):
    name: str = Field(examples=["Arjan"])
    email: EmailStr = Field(
        examples=["user@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )
    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user"
    )
    role: Role = Field(
        default=None, description="The role of the user", examples=[1, 2, 4, 8]
    )

    @field_validator("name")
    @classmethod
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError(
                "Name is invalid, must contain only letters and be at least 2 characters long"
            )
        return v

    @field_validator("role", mode="before")
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f"Role is invalid, please use one of the following: {', '.join([x.name for x in Role])}"
            )

    @model_validator(mode="before")
    @classmethod
    def validate_user(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v:
            raise ValueError("Name and password are required")
        if v["name"].casefold() in v["password"].casefold():
            raise ValueError("Password cannot contain name")
        if not VALID_PASSWORD_REGEX.match(v["password"]):
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest()
        return v

valida é idêntico ao do exemplo anterior.

In [5]:
def validate(data: dict[str, Any]) -> None:
    try:
        user = User.model_validate(data)
        print(user)
    except ValidationError as e:
        print("User is invalid:")
        print(e)


Apenas casos de teste, assim como no anterior, mas agora são vários. O for serve para percorrer todos e printar o resultado da validação de cada caso de teste.

In [7]:
def main() -> None:
    test_data = dict(
        good_data={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Admin",
        },
        bad_role={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Programmer",
        },
        bad_data={
            "name": "Arjan",
            "email": "bad email",
            "password": "bad password",
        },
        bad_name={
            "name": "Arjan<-_->",
            "email": "example@arjancodes.com",
            "password": "Password123",
        },
        duplicate={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Arjan123",
        },
        missing_data={
            "email": "<bad data>",
            "password": "<bad data>",
        },
    )

    for example_name, data in test_data.items():
        print(example_name)
        validate(data)
        print()


if __name__ == "__main__":
    main()

good_data
name='Arjan' email='example@arjancodes.com' password=SecretStr('**********') role=<Role.Admin: 4>

bad_role
User is invalid:
1 validation error for User
role
  Value error, Role is invalid, please use one of the following: Author, Editor, Admin, SuperAdmin [type=value_error, input_value='Programmer', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

bad_data
User is invalid:
1 validation error for User
  Value error, Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number [type=value_error, input_value={'name': 'Arjan', 'email'...ssword': 'bad password'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

bad_name
User is invalid:
1 validation error for User
name
  Value error, Name is invalid, must contain only letters and be at least 2 characters long [type=value_error, input_value='Arjan<-_->', input_type=str]
    For further information visit https://